In [1]:
import sys

if "google.colab" in sys.modules:
    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    original_data = '/content/drive/My Drive/MS1/original_dataset'
    final_data = '/content/drive/My Drive/MS1/Final_Dataset'

    # Install required packages
    !pip install pymatgen torch_geometric mp_api

else:
    original_data = "original_dataset"
    final_data = "Final_Dataset"

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 5.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.1/829.1 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 106.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 94.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 13.1 MB

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader


from pymatgen.core import Structure, PeriodicSite, DummySpecie, Composition, Element
from pymatgen.core.periodic_table import Element as PMGElement
from pymatgen.analysis.local_env import MinimumDistanceNN, CrystalNN, VoronoiNN
from mp_api.client import MPRester
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from sklearn.model_selection import train_test_split

# import json
# import config

API_KEY = "Not relevant right now"  # Replace with your Materials Project API key


# Data Preparation

In [41]:
# Get 1500 data points
# from sklearn.preprocessing import StandardScaler

comb_df = pd.read_csv(f"{final_data}/combined/combined_data.csv")

# s, p = train_test_split(comb_df, test_size=0.5, random_state=42, stratify=comb_df['strata'])

k = comb_df[comb_df["strata"] != 32]
l = k[k["strata"] != 33]

comb_df = l.copy()

In [42]:
# Sample row
sample_row = comb_df.iloc[0]
sample_row

,0
_id,BN_B62C2N63_816498e1-eef8-400f-93d4-17a0a435389a
dataset_material,high_BN
vacancy_sites,1.0
substitution_sites,2.0
defect_sites,3.0
band_gap_value,0.4787
strata,0


In [43]:
# Get defective structure
defective_file_path = f"{original_data}/{sample_row["dataset_material"]}/cifs/{sample_row["_id"]}.cif"
defective_structure = Structure.from_file(defective_file_path)

# Get reference structure
ref_file_path = f"{final_data}/ref_cifs/{sample_row["dataset_material"]}.cif"
reference_structure = Structure.from_file(ref_file_path)

/usr/local/lib/python3.12/dist-packages/pymatgen/core/structure.py:3117: UserWarning: Issues encountered while parsing CIF: 32 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]


## Defective structure to defects cloud

In [44]:
vnn = VoronoiNN(allow_pathological=True)

def struct_to_dict(structure):
    rounded_coords = np.round(structure.frac_coords, 3)
    return {tuple(coord): site for coord, site in zip(rounded_coords, structure.sites)}

# Functions to get formation energy from Materials Project
def get_formation(element, API_KEY):
    with MPRester(API_KEY) as mpr:
        results = mpr.materials.summary.search(
            elements=[element],
            num_elements=1,
            fields= ["energy_per_atom"]
        )
        forms_list = [result.energy_per_atom for result in results]
        avg_formation_energy = np.mean(forms_list)

    return avg_formation_energy


def get_from_json(element, API_KEY):
    try:
        the_dict = {
                "Mo":-10.9332, "S":-4.127, "W":-13.0106, "Se":-3.489,
                "B":-6.704, "N":-8.324, "Ga":-3.03, "In":-2.715,
                "P":-5.362, "V":-8.992, "O":-4.938, "C":-9.226
            }
        if element in the_dict:
            to_return = the_dict[element]

        else:
            to_return = get_formation(element, API_KEY)
            the_dict[element] = to_return


    except:
        the_dict = {}
        with open("./test.json", "a") as f:
            to_return = get_formation(element, API_KEY)
            the_dict[element] = to_return

    return to_return

def fe_site(original, new):
    if new == 0: # For vcancy
        fe_defect = get_from_json(original, API_KEY) * -1

    else: # For substitution
        form_original = get_from_json(original, API_KEY)
        form_new = get_from_json(new, API_KEY)
        fe_defect = (form_original * -1) + form_new

    return fe_defect

def get_ir(e):
    try:
        return float(e.average_ionic_radius)
    except Exception:
        try:
            return float(e.atomic_radius)
        except Exception:
            return 0.0

def get_defects_structure(defective_struct, reference_struct):
    mindnn = MinimumDistanceNN()
    # struct to dict
    defective_dict = struct_to_dict(defective_struct)
    reference_dict = struct_to_dict(reference_struct)

    # Get lattice of defective structure
    structure_lattice = defective_struct.lattice

    # List to add all defect sites
    defects_list = []

    # Dictionary to hold properties of each defect site
    defects_properties = {}

    ref_index = 0

    for ref_coord, ref_site in reference_dict.items():
        # Use the reference coordinates to get the defective site
        ref_index = ref_index + 1

        def_site = defective_dict.get(ref_coord)

        if def_site:  # The site is found in both the reference structure and the defective structure
            # But are the species the same?
            ref_specie = ref_site.specie
            def_specie = def_site.specie
            if ref_specie != def_specie:  # Substitution
                # Add site to defects list
                defects_list.append(def_site)

                # Get atomic number change and defect type
                add_property = {
                    "original_Z":ref_specie.Z,
                    "new_Z": def_specie.Z,
                    "Z_change": def_specie.Z - ref_specie.Z,

                    "original_en": ref_specie.X,
                    "new_en": def_specie.X,
                    "en_change": def_specie.X - ref_specie.X,

                    "original_ir": get_ir(ref_specie),
                    "new_ir": get_ir(ref_specie),
                    "ir_change": get_ir(def_specie) - get_ir(ref_specie),

                    "original_ar": ref_specie.atomic_radius,
                    "new_ar": def_specie.atomic_radius,
                    "ar_change": def_specie.atomic_radius - ref_specie.atomic_radius,

                    "original_row": ref_specie.row,
                    "new_row": def_specie.row,
                    "row_change": def_specie.row - ref_specie.row,

                    "original_group": ref_specie.group,
                    "new_group": def_specie.group,
                    "group_change": def_specie.group - ref_specie.group,

                    "original_max_os": max(ref_specie.common_oxidation_states),
                    "new_max_os": max(def_specie.common_oxidation_states),

                    "original_ef": ref_specie.electron_affinity,
                    "new_ef": def_specie.electron_affinity,

                    "vacancy_defect": 0.0,
                    "substitution_defect": 1.0,
                    "bonds_broken": 0.0,
                    "site_fe": fe_site(ref_site.species_string,def_site.species_string),
                    "ref_idx": ref_index-1
                }

                # "original_ve": int(ref_specie.valence[-1]) if ref_specie.valence else 0,
                # "new_ve": int(def_specie.valence[-1]) if def_specie.valence else 0,
                # "ve_change": int(def_specie.valence[-1]) - int(ref_specie.valence[-1]) if ref_specie.valence or def_specie.valence else 0,

                voro_info = vnn.get_voronoi_polyhedra(reference_struct, add_property["ref_idx"])
                add_property["Voronoi_volume"] = sum(v["volume"] for v in voro_info.values())


                defects_properties[def_site] = add_property

        else: # the site from ref_structure is not found in defective structure
            # This means that the site is a vacancy site
            # Add site to defective structure
            vacant_site = PeriodicSite(
                species= DummySpecie(),
                coords= ref_coord,
                coords_are_cartesian= False,
                lattice= structure_lattice
                )

            # Add site to defects list
            defects_list.append(vacant_site)

            ref_specie = ref_site.specie

            # Get atomic number change and defect type
            add_property={
                "original_Z":ref_specie.Z,
                "new_Z": 0,
                "Z_change": 0 - ref_site.specie.Z,

                "original_en": ref_specie.X,
                "new_en": 0.0,
                "en_change": 0.0 - ref_specie.X,

                "original_ir": get_ir(ref_specie),
                "new_ir": 0.0,
                "ir_change": 0.0 - get_ir(ref_specie),

                "original_ar": ref_specie.atomic_radius,
                "new_ar": 0.0,
                "ar_change": 0.0 - ref_specie.atomic_radius,

                "original_row": ref_specie.row,
                "new_row": 0,
                "row_change": 0 - ref_specie.row,

                "original_group": ref_specie.group,
                "new_group": 0,
                "group_change": 0 - ref_specie.group,

                "original_max_os": max(ref_specie.common_oxidation_states),
                "new_max_os": 0,

                "original_ef": ref_specie.electron_affinity,
                "new_ef": 0.0,

                "vacancy_defect": 1.0,
                "substitution_defect": 0.0,
                "bonds_broken": mindnn.get_cn(reference_struct, reference_struct.sites.index(ref_site)),
                "site_fe": fe_site(ref_site.species_string,0),
                "ref_idx": ref_index-1
            }

            # "original_ve": int(ref_specie.valence[-1]) if ref_specie.valence else 0,
            # "new_ve": 0,
            # "ve_change": 0 - int(ref_specie.valence[-1]) if ref_specie.valence else 0,

            voro_info = vnn.get_voronoi_polyhedra(reference_struct, add_property["ref_idx"])
            add_property["Voronoi_volume"] = sum(v["volume"] for v in voro_info.values())


            defects_properties[vacant_site] = add_property

    # create a defects structure
    defects_struct = Structure.from_sites(defects_list)

    # Add properties to defects structure
    for a_site in defects_struct.sites:
        if a_site in defects_properties.keys():
            a_site.properties.update(defects_properties[a_site])
        else:
            pass

    return defects_struct

defects_structure = get_defects_structure(defective_structure, reference_structure)
print(defects_structure)

Full Formula (X1 C2)
Reduced Formula: XC2
abc   :  20.099428  20.099428  20.000000
angles:  90.000000  90.000000 120.000011
pbc   :       True       True       True
Sites (3)
  #  SP           a         b         c    Voronoi_volume    Z_change    ar_change    bonds_broken    en_change    group_change    ir_change    new_Z    new_ar    new_ef    new_en    new_group    new_ir    new_max_os    new_row    original_Z    original_ar    original_ef    original_en    original_group    original_ir    original_max_os    original_row    ref_idx    row_change    site_fe    substitution_defect    vacancy_defect
---  ----  --------  --------  --------  ----------------  ----------  -----------  --------------  -----------  --------------  -----------  -------  --------  --------  --------  -----------  --------  ------------  ---------  ------------  -------------  -------------  -------------  ----------------  -------------  -----------------  --------------  ---------  ------------  ---------  -

## Nodes

In [45]:
def get_nodes(defects_struct):
    defect_sites = defects_struct.sites

    nodes = []
    for site in defect_sites:
        site_features = [
            site.properties["Voronoi_volume"],
            site.properties["Z_change"],
            site.properties["ar_change"],
            site.properties["en_change"],
            site.properties["group_change"],
            site.properties["ir_change"],
            site.properties["new_Z"],
            site.properties["new_ar"],
            site.properties["new_ef"],
            site.properties["new_en"],
            site.properties["new_group"],
            site.properties["new_ir"],
            site.properties["new_max_os"],
            site.properties["new_row"],
            # site.properties["new_ve"],
            site.properties["original_Z"],
            site.properties["original_ar"],
            site.properties["original_ef"],
            site.properties["original_en"],
            site.properties["original_group"],
            site.properties["original_ir"],
            site.properties["original_max_os"],
            site.properties["original_row"],
            # site.properties["original_ve"],
            site.properties["row_change"],
            site.properties["bonds_broken"],
            site.properties["vacancy_defect"],
            site.properties["substitution_defect"],
            site.properties["site_fe"],
            # site.properties["ve_change"]
        ]
        nodes.append(site_features)

    return nodes

sample001 = get_nodes(defects_structure)
sample001

[[np.float64(54.66608840320199),
  1,
  -0.15000000000000002,
  0.5099999999999998,
  1,
  -0.10999999999999999,
  6,
  0.7,
  1.262122611,
  2.55,
  14,
  0.41,
  4,
  2,
  5,
  0.85,
  0.27972325,
  2.04,
  13,
  0.41,
  3,
  2,
  0,
  0.0,
  0.0,
  1.0,
  -2.522000000000001],
 [np.float64(54.666086945444036),
  1,
  -0.15000000000000002,
  0.5099999999999998,
  1,
  -0.10999999999999999,
  6,
  0.7,
  1.262122611,
  2.55,
  14,
  0.41,
  4,
  2,
  5,
  0.85,
  0.27972325,
  2.04,
  13,
  0.41,
  3,
  2,
  0,
  0.0,
  0.0,
  1.0,
  -2.522000000000001],
 [np.float64(54.66608694544425),
  -7,
  -0.65,
  -3.04,
  -15,
  -0.63,
  0,
  0.0,
  0.0,
  0.0,
  0,
  0.0,
  0,
  0,
  7,
  0.65,
  -0.07,
  3.04,
  15,
  0.63,
  5,
  2,
  -2,
  3,
  1.0,
  0.0,
  8.324]]

##  Edge index and attributes

In [46]:
def get_edges_and_features(reference_structure, defects_structure):
    a_lat = float(reference_structure.lattice.a)

    from_edge = []
    to_edge = []
    edges = []
    edge_features = []

    cart_coords_ds = defects_structure.cart_coords

    for i, site_i in enumerate(defects_structure):
        for j, site_j in enumerate(defects_structure):
            if j > i :
                # pass
            # else:
                from_edge.append(i)
                to_edge.append(j)

                cart_i = cart_coords_ds[i]
                cart_j = cart_coords_ds[j]
                r_vec = cart_j - cart_i
                r_ij = float(np.linalg.norm(r_vec))

                # if r_ij > 12.0 or r_ij<1e-6:
                    # continue

                dist_angstrom = r_ij
                dist_norm = site_i.distance(site_j)
                dist_lattice_units = r_ij/a_lat

                # Formation energy interaction
                fe_site_i = site_i.properties["site_fe"]
                fe_site_j = site_j.properties["site_fe"]

                fe_product = fe_site_i * fe_site_j
                fe_sum = fe_site_i + fe_site_j
                fe_diff = abs(fe_site_i - fe_site_j)

                # Electrostatic interaction
                q_i = site_i.properties["new_max_os"]
                q_j = site_j.properties["new_max_os"]

                if site_i.properties["vacancy_defect"] == 1:
                    q_i = -q_i

                if site_j.properties["vacancy_defect"] == 1:
                    q_j = -q_j

                charge_product = q_i * q_j
                screened_coulomb = (q_i * q_j) / (r_ij) if r_ij > 0 else 0.0

                # Elastic size interaction
                ir_change_i = site_i.properties["ir_change"]
                ir_change_j = site_j.properties["ir_change"]

                ir_change_product = ir_change_i * ir_change_j
                elastic_size_interaction = (ir_change_i * ir_change_j) / (r_ij ** 3) if r_ij > 0 else 0.0

                # Angular factor
                if r_ij > 0:
                    cos_theta = r_vec[2]/ r_ij
                    angular_factor = 1.0-3.0 * cos_theta ** 2
                else:
                    angular_factor = 0.0

                # Defect interaction
                if site_i.properties["vacancy_defect"] == 1 and site_j.properties["substitution_defect"] == 1:
                    vac_sub = 1
                    vac_vac = 0
                    sub_sub = 0

                if site_i.properties["substitution_defect"] == 1 and site_j.properties["vacancy_defect"] == 1:
                    vac_sub = 1
                    vac_vac = 0
                    sub_sub = 0


                if site_i.properties["substitution_defect"] == 1 and site_j.properties["substitution_defect"] == 1:
                    vac_sub = 0
                    vac_vac = 0
                    sub_sub = 1

                if site_i.properties["vacancy_defect"] == 1 and site_j.properties["vacancy_defect"] == 1:
                    vac_sub = 0
                    vac_vac = 1
                    sub_sub = 0

                edge_features.append([
                    dist_angstrom, dist_norm, dist_lattice_units,
                    fe_product, fe_sum, fe_diff,
                    charge_product, screened_coulomb, ir_change_product,
                    elastic_size_interaction, angular_factor,
                    vac_vac, sub_sub, vac_sub
                    ])

    edges.append(from_edge)
    edges.append(to_edge)

    return edges, edge_features

edgesss, featuresss = get_edges_and_features(reference_structure, defects_structure)

## Global attributes

In [47]:
"""# Host global
host_csv = pd.read_csv("original_dataset/initial_structures.csv")

unique_materials = comb_df["dataset_material"].unique()

all_g = []
for material in unique_materials:
    # Get pristine structure
    pristine_path = f"{final_data}/ref_cifs/{material}.cif"
    pristine = Structure.from_file(pristine_path)


    the_material = material
    g = {}

    # ---- Symmetry ----
    sga = SpacegroupAnalyzer(pristine)
    g["sgn"] = sga.get_space_group_number()
    crystal_systems = {
        "triclinic": 0, "monoclinic": 1, "orthorhombic": 2,
        "tetragonal": 3, "trigonal": 4, "hexagonal": 5, "cubic": 6
    }
    cs = sga.get_crystal_system()
    for name, idx in crystal_systems.items():
        g[f"crystal_system_{name}"] = int(cs == name)

    # ---- Lattice ----
    lat = pristine.lattice
    g["lattice_a"] = float(lat.a)
    g["lattice_b"] = float(lat.b)
    g["lattice_c"] = float(lat.c)
    g["lattice_alpha"] = float(lat.alpha)
    g["lattice_beta"] = float(lat.beta)
    g["lattice_gamma"] = float(lat.gamma)
    g["volume_per_atom"] = float(pristine.volume / len(pristine))
    g["density"] = float(pristine.density)

    # ---- Composition ----
    g["n_species"] = len(pristine.composition.elements)
    # Host composition vector: mean electronegativity, mean atomic radius, etc.
    elems = pristine.composition.elements
    ens = [e.X for e in elems]
    ars = [e.atomic_radius for e in elems]
    g["host_mean_electronegativity"] = float(np.mean(ens)) if ens else 0.0
    g["host_electronegativity_spread"] = float(np.max(ens) - np.min(ens)) if ens else 0.0
    g["host_mean_atomic_radius"] = float(np.mean(ars)) if ars else 0.0

    # split material to get the second part after "_"
    material_parts = material.split("_")
    g["Energy"] = host_csv.loc[host_csv["base"] == material_parts[1], "energy"].values[0]
    g["E_1"] = host_csv.loc[host_csv["base"] == material_parts[1], "E_1"].values[0]
    g["E_VBM"] = host_csv.loc[host_csv["base"] == material_parts[1], "E_VBM"].values[0]

    # List of values
    g_list = list(g.values())

    new_g = {the_material: g_list}

    all_g.append(new_g)

all_g
"""

'# Host global\nhost_csv = pd.read_csv("original_dataset/initial_structures.csv")\n\nunique_materials = comb_df["dataset_material"].unique()\n\nall_g = []\nfor material in unique_materials:\n    # Get pristine structure\n    pristine_path = f"{final_data}/ref_cifs/{material}.cif"\n    pristine = Structure.from_file(pristine_path)\n\n\n    the_material = material\n    g = {}\n\n    # ---- Symmetry ----\n    sga = SpacegroupAnalyzer(pristine)\n    g["sgn"] = sga.get_space_group_number()\n    crystal_systems = {\n        "triclinic": 0, "monoclinic": 1, "orthorhombic": 2,\n        "tetragonal": 3, "trigonal": 4, "hexagonal": 5, "cubic": 6\n    }\n    cs = sga.get_crystal_system()\n    for name, idx in crystal_systems.items():\n        g[f"crystal_system_{name}"] = int(cs == name)\n\n    # ---- Lattice ----\n    lat = pristine.lattice\n    g["lattice_a"] = float(lat.a)\n    g["lattice_b"] = float(lat.b)\n    g["lattice_c"] = float(lat.c)\n    g["lattice_alpha"] = float(lat.alpha)\n    g["l

In [48]:
def get_host_global(dataset_material):
    all_g = [
        {'high_BN': [187,0,0,0,0,0,1,0,20.09942768,20.099427680000005,20.0,90.0,90.0,120.00001201,54.666086945443936,0.37693168428045426,2,2.54,1.0,0.75,-1124.4346,-22.095,-4.663]},
        {'high_P': [53,0,0,1,0,0,0,0,19.799508,27.605556,20.0,90.0,90.0,90.0,75.91339262033999,0.6775239537726686,1,2.19,0.0,1.0,-772.1556,-17.222,-2.252]},
        {'high_InSe': [187,0,0,0,0,0,1,0,24.57884112,24.578841120000003,20.0,90.0,90.0,119.99999231,72.66427979662369,2.2141273580208995,2,2.165,0.7699999999999998,1.35,-522.7948,-16.344,-2.158]},
        {'high_GaSe': [187,0,0,0,0,0,1,0,22.91440026,22.91440026,20.0,90.0,90.0,120.00001238000002,63.156066145091955,1.9546335407989874,2,2.1799999999999997,0.7399999999999998,1.225,-554.0446,-17.288,-2.832]},
        {'high_MoS2': [187,0,0,0,0,0,1,0,25.5225256,25.5225256,20.0,90.0,90.0,119.99999999999999,58.7633701112532,1.5077560973715214,2,2.37,0.41999999999999993,1.225,-1398.1131,-61.587,-0.709]},
        {'high_WSe2': [187,0,0,0,0,0,1,0,26.61655608,26.616556079999995,20.0,90.0,90.0,119.99999999999999,63.90916176361329,2.9599607554087743,2,2.455,0.18999999999999995,1.25,-1384.9085,-15.439,-0.958]},
        {'low_MoS2': [187,0,0,0,0,0,1,0,25.5225256,25.5225256,14.879004,90.0,90.0,119.99999999999999,43.71702094694084,2.026689551762365,2,2.37,0.41999999999999993,1.225,-1398.1131,-61.587,-0.709]},
        {'low_WSe2': [187,0,0,0,0,0,1,0,26.61655608,26.616556079999995,15.068951,90.0,90.0,119.99999999999999,48.15220135334811,3.928555817068852,2,2.455,0.18999999999999995,1.25,-1384.908,-15.439,-0.958]}
    ]
    for g in all_g:
        if dataset_material in g:
            return g[dataset_material]
    return np.zeros(27)

get_host_global("high_P")

[53,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 19.799508,
 27.605556,
 20.0,
 90.0,
 90.0,
 90.0,
 75.91339262033999,
 0.6775239537726686,
 1,
 2.19,
 0.0,
 1.0,
 -772.1556,
 -17.222,
 -2.252]

In [49]:
def get_defective_global(dataset_material, _id):
    focus_csv = pd.read_csv(f"{original_data}/{dataset_material}/defects.csv")

    energy = focus_csv.loc[focus_csv["_id"] == _id, "energy"].values[0]
    f_l = focus_csv.loc[focus_csv["_id"] == _id, "fermi_level"].values[0]
    try:
        t_mag = focus_csv.loc[focus_csv["_id"] == _id, "total_mag"].values[0]
    except KeyError:
        t_mag = 0.0

    return [energy, f_l, t_mag]


get_defective_global(sample_row["dataset_material"], sample_row["_id"])

[np.float64(-1106.8694971), np.float64(-1.24184762), np.float64(0.9996579)]

In [50]:
def get_globals(pristine, defects_structure):
    g = {}

    # ---- Defect configuration summary ----
    g["n_defects"] = len(defects_structure)
    g["n_atoms_pristine"] = len(pristine)
    g["defect_concentration"] = len(defects_structure) / len(pristine)

    vacs = 0
    subs = 0
    for def_site in defects_structure:
        if def_site.properties["vacancy_defect"] == 1:
            vacs += 1
        else:
            subs+=1


    g["n_vacancy"] = vacs
    g["n_substitution"] = subs

    return list(g.values())

## Nodes + Edges + Global attributes

In [51]:
# Create graph representation of the structures

def graphy(row):
    defective_structure = Structure.from_file(f"{original_data}/{row["dataset_material"]}/cifs/{row["_id"]}.cif")
    reference_structure = Structure.from_file(f"{final_data}/ref_cifs/{row["dataset_material"]}.cif")

    defects_only_structure = get_defects_structure(defective_structure, reference_structure)

    # nodes, edges, edge_features, ids, ratios = to_graph.get_c_graph(defects_only_structure)
    # nodes, edges, edge_features, global_features = to_graph.get_c_graph(defects_only_structure)
    nodes = get_nodes(defects_only_structure)
    edges, edge_features = get_edges_and_features(reference_structure, defects_only_structure)

    # Global
    d_m = row["dataset_material"]

    global_features1 = get_globals(reference_structure, defects_only_structure)
    global_features2 = get_host_global(d_m)
    global_features3 = get_defective_global(d_m, row["_id"])

    global_features = global_features1 + global_features2 + global_features3



    # Scale the target variable
    target = row["band_gap_value"]

    the_data = Data(
        x=torch.tensor(nodes, dtype=torch.float),
        edge_index=torch.tensor(edges, dtype=torch.long),
        edge_attr=torch.tensor(edge_features, dtype=torch.float),
        u=torch.tensor(global_features, dtype=torch.float).unsqueeze(0),
        y=torch.tensor(target, dtype=torch.float).unsqueeze(0)
    )
    return the_data

In [52]:
# Split the data
train_set, test_set = train_test_split(comb_df, test_size=0.2, stratify=comb_df['strata'], random_state=42)
test_set, val_set = train_test_split(test_set, test_size=0.5, random_state=42)

# Turn each dataset into graph data
train_graphs = train_set.apply(lambda row: graphy(row), axis = 1).tolist()
val_graphs = val_set.apply(lambda row: graphy(row), axis = 1).tolist()
test_graphs = test_set.apply(lambda row: graphy(row), axis = 1).tolist()

# Create data loaders
train_loader = DataLoader(train_graphs, batch_size=1, shuffle=True)
val_loader = DataLoader(val_graphs, batch_size=1, shuffle=False)
test_loader = DataLoader(test_graphs, batch_size=1, shuffle=False)

Streaming output truncated to the last 5000 lines.
/usr/local/lib/python3.12/dist-packages/pymatgen/core/structure.py:3117: UserWarning: Issues encountered while parsing CIF: 47 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/usr/local/lib/python3.12/dist-packages/pymatgen/core/structure.py:3117: UserWarning: Issues encountered while parsing CIF: 48 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/usr/local/lib/python3.12/dist-packages/pymatgen/core/structure.py:3117: UserWarning: Issues encountered while parsing CIF: 46 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/usr/local/lib/python3.12/dist-packages/pymatgen/core/structure.py:3117: UserWarning: Issues encountered while parsing CIF: 48 fracti

# The GNN Model

In [53]:
# import dependancies
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import global_mean_pool, NNConv, CGConv, global_max_pool, GCNConv


In [54]:
# Train model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for data in loader:
            data = data.to(device)

            out = model(data)
            loss = loss_fn(out, data.y)

            total_loss += loss.item() * data.num_graphs
    return total_loss / len(loader.dataset)

def train(model, train_loader, val_loader, epochs, optimizer, loss_fn):
    train_losses = []
    val_losses = []

    # Learning rate scheduler to help convergence
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)

    for epoch in range(1, epochs+1):
        model.train()
        total_loss = 0
        for data in train_loader:
            data = data.to(device)
            optimizer.zero_grad()

            out = model(data)
            loss = loss_fn(out, data.y)

            loss.backward()
            optimizer.step()
            total_loss += loss.item() * data.num_graphs
        train_loss = total_loss / len(train_loader.dataset)
        # train_loss = train(the_model, train_loader, optimizer, loss_fn)
        val_loss = evaluate(model, val_loader, loss_fn)
        scheduler.step()

        print(f'Epoch {epoch:03d}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')

        train_losses.append(train_loss)
        val_losses.append(val_loss)

    # Test model
    test_loss = evaluate(model, test_loader, loss_fn)
    print(f'Test Loss: {test_loss:.4f}')

    return train_losses, val_losses

def predict(model, loader, actuals=False):
    model.eval()
    if actuals:
        predictions = []
        actual = []
        with torch.no_grad():
            for data in loader:
                data = data.to(device)

                out = model(data)
                predictions.append(out.cpu().numpy())
                actual.append(data.y.cpu().numpy())

        predictions = np.concatenate(predictions, axis=0).flatten()
        actual = np.concatenate(actual, axis=0).flatten()
        return predictions, actual
    else:
        predictions = []
        with torch.no_grad():
            for data in loader:
                data = data.to(device)

                out = model(data)
                predictions.append(out.cpu().numpy())

        predictions = np.concatenate(predictions, axis=0).flatten()
        return predictions

# Informative Graphs
def get_graphs(train_losses, val_losses, the_model):
    fig, axs = plt.subplots(3, 1, figsize=(10, 10))

    axs[0].plot(train_losses, label='Train Loss')
    axs[0].plot(val_losses, label='Validation Loss')
    axs[0].set_xlabel('Epoch')
    axs[0].set_ylabel('Loss')
    axs[0].set_title('Training  and Validation Loss')
    axs[0].legend()

    # model values
    predicted_values, actual_values = predict(the_model, test_loader, actuals=True)


    axs[1].scatter(actual_values, predicted_values )
    axs[1].plot(actual_values, actual_values, c="black")
    axs[1].set_xlabel('Actual Band Gaps(eV)')
    axs[1].set_ylabel('Predicted Band Gaps(eV)')
    axs[1].set_title('Actual vs Predicted Band Gap Values')

    axs[2].plot(actual_values[100:150], label='Actual Values', marker='o')
    axs[2].plot(predicted_values[100:150], label='Predicted Values', marker='o')
    axs[2].set_xlabel('Samples')
    axs[2].set_ylabel('Band gap(eV)')
    axs[2].set_title('Actual vs Predicted Band Gap Values')
    axs[2].legend()

    plt.show()


class LogCoshLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, input, target):
        ey_t = input-target
        return torch.mean(torch.log(torch.cosh(ey_t + 1e-12)))


class RelativeMSELoss(nn.Module):
    def __inti__(self):
        super().__init__()

    def forward(self, y_pred, y_true):
        return torch.mean(((y_pred-y_true)/ (y_true + 1e-8)) ** 2)


class SMAPELoss(nn.Module):
    def __inti__(self):
        super().__init__()

    def forward(self, y_pred, y_true):
        numerator = torch.abs(y_pred - y_true)
        denominator = (torch.abs(y_true) + torch.abs(y_pred))/2.0 + 1e-8
        return torch.mean(numerator/denominator)

# Loss function
loss_fn = nn.MSELoss()
# loss_fn = LogCoshLoss()
# loss_fn = RelativeMSELoss()
# loss_fn = SMAPELoss()

## Model 1

In [55]:
class GNNModelCG(nn.Module):
    def __init__(self, node_dim=8, edge_dim=3,hidden_dim=64, embed_dim=64, u_dim=6):
        super().__init__()
        self.conv1 = CGConv(node_dim, hidden_dim, aggr='mean')
        self.conv2 = CGConv(hidden_dim, edge_dim, aggr='mean')

        self.global_embed = nn.Linear(u_dim, embed_dim)

        self.fc = nn.Sequential(
            nn.Linear(hidden_dim + embed_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, data):
        x, edge_index, edge_attr, batch, u = (
            data.x,
            data.edge_index,
            data.edge_attr,
            data.batch,
            data.u,
        )
        print(x.shape, edge_index.shape, edge_attr.shape)
        x = F.relu(self.conv1(x, edge_index, edge_attr))

        x = F.relu(self.conv2(x, edge_index, edge_attr))

        x = global_mean_pool(x, batch)

        u = self.global_embed(u)

        x = torch.cat([x, u], dim=1)
        the_return = self.fc(x)
        return the_return

# Use the CGConv-only model if you want a simpler graph architecture.
GNN_model_cg = GNNModelCG().to(device)

# optimizer
optimizer = torch.optim.Adam(GNN_model_cg.parameters(), lr=0.001)

# Train the model
train_losses, val_losses = train(GNN_model_cg, train_loader, val_loader, epochs=20, optimizer=optimizer, loss_fn=loss_fn)
get_graphs(train_losses, val_losses, GNN_model_cg)

torch.Size([9, 27]) torch.Size([2, 36]) torch.Size([36, 14])


RuntimeError: mat1 and mat2 shapes cannot be multiplied (36x68 and 80x8)

## Model 2

In [56]:
class Model2(nn.Module):
    def __init__(self, node_dim=27, edge_dim=14, hidden_dim=64, embed_dim=64, u_dim=25):
        super().__init__()

        # For the edges
        self.edge_nn = nn.Sequential(
            nn.Linear(edge_dim, 64),
            nn.ReLU(),
            nn.Linear(64, node_dim * hidden_dim)
        )

        self.conv1 = NNConv(node_dim, hidden_dim, self.edge_nn, aggr='mean')

        # self.conv1 = CGConv(hidden_dim, hidden_dim, aggr='mean')
        self.conv2 = CGConv(hidden_dim, edge_dim, aggr='mean')

        self.global_embed = nn.Linear(u_dim, embed_dim)

        self.fc = nn.Sequential(
            nn.Linear(hidden_dim + embed_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, data):
        x, edge_index, edge_attr, batch, u = (
            data.x,
            data.edge_index,
            data.edge_attr,
            data.batch,
            data.u,
        )
        x = F.relu(self.conv1(x, edge_index, edge_attr))
        x = F.relu(self.conv2(x, edge_index, edge_attr))

        x = global_mean_pool(x, batch)

        u = self.global_embed(u)

        x = torch.cat([x, u], dim=1)
        the_return = self.fc(x)
        return the_return

# Use the CGConv-only model if you want a simpler graph architecture.
model_2 = Model2().to(device)

# optimizer
optimizer = torch.optim.Adam(model_2.parameters(), lr=0.00001)

# Train this model
train_losses, val_losses = train(model_2, train_loader, val_loader, epochs=20, optimizer=optimizer, loss_fn=loss_fn)
get_graphs(train_losses, val_losses, model_2)

RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x31 and 25x64)

## Model 3

In [ ]:
class GNNModel(nn.Module):
    def __init__(self, node_dim=27, edge_dim=14, hidden_dim=128, embed_dim=64, global_dim=25):
        super().__init__()

        # Edge network for NNConv: maps each edge_attr to a weight matrix
        self.edge_mlp = nn.Sequential(
            nn.Linear(edge_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, node_dim * hidden_dim),
        )

        # First message-passing layer
        self.conv1 = NNConv(node_dim, hidden_dim, self.edge_mlp, aggr='mean')

        # Second message-passing layer using CGConv
        self.conv2 = CGConv(hidden_dim, edge_dim, aggr='mean')
        # self.conv3 = CGConv(hidden_dim, edge_dim, aggr='mean')

        # Embed the graph-level global features
        self.global_embed = nn.Linear(global_dim, embed_dim)

        # Final MLP head
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim + embed_dim, 128),
            nn.Dropout(0.2),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.Dropout(0.2),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.Dropout(0.2),
            # NO ReLU here - allow both positive and negative outputs
            nn.Linear(32, 1)
        )

    def forward(self, data):
        x, edge_index, edge_attr, batch, u = (
            data.x,
            data.edge_index,
            data.edge_attr,
            data.batch,
            data.u,
        )

        x = F.relu(self.conv1(x, edge_index, edge_attr))
        x = F.relu(self.conv2(x, edge_index, edge_attr))
        # x = F.relu(self.conv3(x, edge_index, edge_attr))

        x = global_mean_pool(x, batch)
        u = self.global_embed(u)

        x = torch.cat([x, u], dim=1)
        return self.fc(x)

GNN_model = GNNModel().to(device)

# optimizer with L2 regularization to prevent overfitting
optimizer = torch.optim.Adam(GNN_model.parameters(), lr=0.001, weight_decay=1e-4)

train_losses, val_losses = train(GNN_model, train_loader, val_loader, epochs=20, optimizer=optimizer, loss_fn=loss_fn)
get_graphs(train_losses, val_losses, GNN_model)

# CVAE

## Data Preparation

In [57]:
# Defects cloud to padded tensor
def cloud_to_tensor(defect_cloud, max_points = 25):
    rows = []
    for a_site in defect_cloud:
        fc = a_site.frac_coords.tolist()

        required = [
            a_site.properties["original_Z"]/94,
            a_site.properties["new_Z"]/94,
            a_site.properties["substitution_defect"],
            a_site.properties["vacancy_defect"]
        ]

        combined = fc + required

        rows.append(combined)

    while len(rows) < max_points:
        rows.append([0.0]*7)

    return torch.tensor(rows, dtype=torch.float32)

# Test
test_tensor = cloud_to_tensor(defects_structure)
test_tensor

tensor([[0.1667, 0.7083, 0.1927, 0.0532, 0.0638, 1.0000, 0.0000],
        [0.9167, 0.8333, 0.1927, 0.0532, 0.0638, 1.0000, 0.0000],
        [0.9580, 0.7920, 0.1930, 0.0745, 0.0000, 0.0000, 1.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0

In [59]:
def tensor_to_cloud(tensor_cloud, threshold = 0.1):

    points = []
    for row in tensor_cloud:
        row = row.detach().cpu()
        if row.norm() < threshold:
            continue  # padding row
        frac_coords = row[:3].clamp(0.0, 1.0).numpy()

        Zp = int((row[3] * 94).round().clamp(0, 94).item())
        Zd = int((row[4] * 94).round().clamp(0, 94).item())

        sub_defect = row[5].item()
        vac_defect = row[6].item()

        if sub_defect:
            defect_type = "substitution"
        elif vac_defect:
            defect_type = "vacancy"

        points.append({
            "fractional_coords": frac_coords,
            "Z_pristine":  Zp,
            "Z_defective": Zd,
            "defect_type":  defect_type,
        })
    return points

test_cloud = tensor_to_cloud(test_tensor)
test_cloud


[{'fractional_coords': array([0.166667  , 0.708333  , 0.19268163], dtype=float32),
  'Z_pristine': 5,
  'Z_defective': 6,
  'defect_type': 'substitution'},
 {'fractional_coords': array([0.916667  , 0.833333  , 0.19268163], dtype=float32),
  'Z_pristine': 5,
  'Z_defective': 6,
  'defect_type': 'substitution'},
 {'fractional_coords': array([0.958, 0.792, 0.193], dtype=float32),
  'Z_pristine': 7,
  'Z_defective': 0,
  'defect_type': 'vacancy'}]

In [60]:
def func_1(reference_structure, defects_structure, band_gap_value):
    # Node features for each atom in pristine structure
    node_feats = []
    for site in reference_structure.sites:
        el = site.specie
        node_feats.append([
            el.Z / 94.0,
            el.X / 4.0 if el.X else 0.0,
            el.atomic_radius if el.atomic_radius else 0.0,
            el.row / 9.0,
            el.group / 18.0,
        ])
    node_features = torch.tensor(node_feats, dtype=torch.float32)


    # Build edge index from radius graph with cutoff of 5 Å
    src, dst = [], []
    coords = np.array([s.coords for s in reference_structure.sites])
    for i in range(len(reference_structure.sites)):
        for j in range(len(reference_structure.sites)):
            if i == j:
                continue
            dist = np.linalg.norm(coords[i] - coords[j])
            if dist < 5.0:
                src.append(i)
                dst.append(j)
    edge_index = torch.tensor([src, dst], dtype=torch.long)

    # Defect cloud as tensor
    tensor_cloud = cloud_to_tensor(defects_structure)

    # band_gap
    band_gap = torch.tensor(band_gap_value, dtype=torch.float32)

    return {
        "node_features": node_features,
        "edge_index":    edge_index,
        "defect_cloud":  tensor_cloud,
        "band_gap":      band_gap
    }

band_gap_value = sample_row["band_gap_value"]
output = func_1(reference_structure, defects_structure, band_gap_value)
output

{'node_features': tensor([[0.0532, 0.5100, 0.8500, 0.2222, 0.7222],
         [0.0532, 0.5100, 0.8500, 0.2222, 0.7222],
         [0.0532, 0.5100, 0.8500, 0.2222, 0.7222],
         [0.0532, 0.5100, 0.8500, 0.2222, 0.7222],
         [0.0532, 0.5100, 0.8500, 0.2222, 0.7222],
         [0.0532, 0.5100, 0.8500, 0.2222, 0.7222],
         [0.0532, 0.5100, 0.8500, 0.2222, 0.7222],
         [0.0532, 0.5100, 0.8500, 0.2222, 0.7222],
         [0.0532, 0.5100, 0.8500, 0.2222, 0.7222],
         [0.0532, 0.5100, 0.8500, 0.2222, 0.7222],
         [0.0532, 0.5100, 0.8500, 0.2222, 0.7222],
         [0.0532, 0.5100, 0.8500, 0.2222, 0.7222],
         [0.0532, 0.5100, 0.8500, 0.2222, 0.7222],
         [0.0532, 0.5100, 0.8500, 0.2222, 0.7222],
         [0.0532, 0.5100, 0.8500, 0.2222, 0.7222],
         [0.0532, 0.5100, 0.8500, 0.2222, 0.7222],
         [0.0532, 0.5100, 0.8500, 0.2222, 0.7222],
         [0.0532, 0.5100, 0.8500, 0.2222, 0.7222],
         [0.0532, 0.5100, 0.8500, 0.2222, 0.7222],
         [0.05

In [61]:
sample_dataset = comb_df.sample(n=500, random_state=42).reset_index(drop=True)

sample_train, sample_val = train_test_split(sample_dataset, test_size=0.3, random_state=42)
sample_train.reset_index(drop=True, inplace=True)
sample_val.reset_index(drop=True, inplace=True)

# Create data for cvae
def cvae_data(row):
    defective_structure = Structure.from_file(f"{original_data}/{row["dataset_material"]}/cifs/{row["_id"]}.cif")
    reference_structure = Structure.from_file(f"{final_data}/ref_cifs/{row["dataset_material"]}.cif")

    defects_only_structure = get_defects_structure(defective_structure, reference_structure)

    band_gap_value = row["band_gap_value"]
    output = func_1(reference_structure, defects_only_structure, band_gap_value)

    the_data = Data(
        node_features = output["node_features"],
        edge_index = output["edge_index"],
        defect_cloud = output["defect_cloud"],
        band_gap = output["band_gap"]
    )
    return the_data

train_dataset = sample_train.apply(lambda row: cvae_data(row), axis = 1).tolist()
val_dataset = sample_val.apply(lambda row: cvae_data(row), axis = 1).tolist()

/usr/local/lib/python3.12/dist-packages/pymatgen/core/structure.py:3117: UserWarning: Issues encountered while parsing CIF: 23 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/usr/local/lib/python3.12/dist-packages/pymatgen/core/structure.py:3117: UserWarning: Issues encountered while parsing CIF: 24 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/usr/local/lib/python3.12/dist-packages/pymatgen/core/structure.py:3117: UserWarning: Issues encountered while parsing CIF: 47 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/usr/local/lib/python3.12/dist-packages/pymatgen/core/structure.py:3117: UserWarning: Issues encountered while parsing CIF: 48 fractional coordinates rounded to ideal values to avoid i

In [62]:
def collate_fn(batch):
    batch_node_features = []
    batch_edge_index = []
    batch_defect_cloud = []
    batch_band_gap = []

    node_offset = 0
    for item in batch:
        batch_node_features.append(item["node_features"])
        batch_defect_cloud.append(item["defect_cloud"])
        batch_band_gap.append(item["band_gap"])

        edge_index = item["edge_index"] + node_offset
        batch_edge_index.append(edge_index)

        node_offset += item["node_features"].shape[0]

    return {
        "node_features": torch.cat(batch_node_features, dim=0),
        "edge_index": torch.cat(batch_edge_index, dim=1),
        "defect_cloud": torch.stack(batch_defect_cloud, dim=0),
        "band_gap": torch.stack(batch_band_gap, dim=0)
    }

In [75]:
test_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)

for batch in test_loader:
    print(batch)
    break

DataBatch(edge_index=[2, 21498], node_features=[1296, 5], defect_cloud=[200, 7], band_gap=[8], batch=[1296], ptr=[9])


## The model architecture

In [80]:
#  ENCODER
class PristineGNNEncoder(nn.Module):
    def __init__(self, node_dim = 5, hidden_dim = 128, latent_dim = 32, n_max = 25, point_dim = 7):
        super().__init__()

        self.conv1 = GCNConv(node_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)

        cloud_flat = n_max * point_dim
        self.fc_combine = nn.Linear(hidden_dim * 2 + cloud_flat, hidden_dim)

        self.fc_mu      = nn.Linear(hidden_dim, latent_dim)
        self.fc_log_var = nn.Linear(hidden_dim, latent_dim)

    # def forward(self, data):
    def forward(self,
            node_features: torch.Tensor,  # (total_nodes, node_dim)
            edge_index:    torch.Tensor,  # (2, total_edges)
            defect_cloud:  torch.Tensor,  # (B * n_max, point_dim)
            batch:         torch.Tensor   # (total_nodes,) – node→graph mapping
            ) -> tuple[torch.Tensor, torch.Tensor]:
        '''node_features = data.node_features
        edge_index = data.edge_index
        defect_cloud = data.defectt_cloud
        batch = data.batch '''

        h = F.relu(self.conv1(node_features, edge_index))
        h = F.relu(self.conv2(h, edge_index))

        h_mean = global_mean_pool(h, batch)
        h_max  = global_max_pool(h, batch)
        h_graph = torch.cat([h_mean, h_max], dim=-1)

        B = h_graph.shape[0]
        cloud_flat = defect_cloud.view(B, -1)  # (B, n_max*point_dim)

        combined = torch.cat([h_graph, cloud_flat], dim=-1)
        hidden   = F.relu(self.fc_combine(combined))

        mu      = self.fc_mu(hidden)       # (B, latent_dim)
        log_var = self.fc_log_var(hidden)  # (B, latent_dim)
        return mu, log_var


# DECODER
class DefectCloudDecoder(nn.Module):
    def __init__(self,hidden_dim = 128, latent_dim = 32, n_max = 25, point_dim = 7):
        super().__init__()
        self.n_max     = n_max
        self.point_dim = point_dim

        # +1 for band gap conditioning
        self.net = nn.Sequential(
            nn.Linear(latent_dim + 1, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.ReLU(),
            nn.Linear(hidden_dim * 2, n_max * point_dim),
        )

    def forward(self, z, band_gap):

        if band_gap.dim() == 1:
            band_gap = band_gap.unsqueeze(-1)

        inp = torch.cat([z, band_gap], dim=-1)
        out = self.net(inp)
        out = out.view(-1, self.n_max, self.point_dim)

        # Apply activations per channel
        coords  = torch.sigmoid(out[..., :3])
        z_vals  = torch.sigmoid(out[..., 3:5])
        logits  = out[..., 5:]

        return torch.cat([coords, z_vals, logits], dim=-1)


# ACTUAL MODEL(ENCODER + DECODER)
class DefectCVAE(nn.Module):
    def __init__(self, node_dim = 5, hidden_dim = 128, latent_dim = 32, n_max = 25, point_dim = 7):

        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = PristineGNNEncoder(node_dim, hidden_dim, latent_dim, n_max, point_dim)
        self.decoder = DefectCloudDecoder(hidden_dim,latent_dim, n_max, point_dim)

    def reparameterize(self, mu, log_var):
        if self.training:
            std = torch.exp(0.5 * log_var)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu

    def forward(self, data):
        mu, log_var = self.encoder(data.node_features, data.edge_index, data.defect_cloud, data.batch)
        z = self.reparameterize(mu, log_var)
        recon = self.decoder(z, data.band_gap)
        return recon, mu, log_var

    @torch.no_grad()
    def generate(self, target_band_gap, n_samples, device = "cpu"):
        self.eval()
        z = torch.randn(n_samples, self.latent_dim, device=device)
        bg = torch.full((n_samples,), target_band_gap,
                        dtype=torch.float32, device=device)
        cloud_tensors = self.decoder(z, bg)

        results = []
        for i in range(n_samples):
            points = tensor_to_cloud(cloud_tensors[i])
            results.append(points)
        return results

In [81]:
# Loss function for cvae
def cvae_loss(recon_cloud, target_cloud, mu, log_var, beta=1e-3):
    # Continuous features: coords + Z values
    mse_loss = F.mse_loss(recon_cloud[..., :5], target_cloud[..., :5])

    # Type logits: everything from index 5 onward (works for 2 or 4 channels)
    n_type_channels = recon_cloud.shape[-1] - 5
    recon_logits   = recon_cloud[..., 5:]
    target_classes = target_cloud[..., 5:].argmax(dim=-1)

    mask = (target_cloud.norm(dim=-1) > 0.01)
    if mask.any():
        ce_loss = F.cross_entropy(recon_logits[mask], target_classes[mask])
    else:
        ce_loss = torch.tensor(0.0, requires_grad=True)

    kl_loss = -0.5 * torch.mean(1 + log_var - mu.pow(2) - log_var.exp())

    total = mse_loss + ce_loss + beta * kl_loss
    return {"loss": total, "mse": mse_loss, "ce": ce_loss, "kl": kl_loss}

In [82]:
#  TRAINING LOOP

def train_one_epoch(model:      DefectCVAE,
                    loader:     DataLoader,
                    optimizer:  torch.optim.Optimizer,
                    device:     str,
                    beta:       float = 1e-3) -> dict[str, float]:
    model.train()
    totals = {"loss": 0.0, "mse": 0.0, "ce": 0.0, "kl": 0.0}

    for batch in loader:
        # Move tensors to device
        # batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()
        recon, mu, log_var = model(batch)

        B = batch.band_gap.shape[0]
        target_cloud = batch.defect_cloud.view(B, -1, batch.defect_cloud.shape[-1])
        losses = cvae_loss(recon, target_cloud, mu, log_var, beta)
        # losses = cvae_loss(recon, batch["defect_cloud"], mu, log_var, beta)
        losses["loss"].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        for k in totals:
            totals[k] += losses[k].item()

    n = max(len(loader), 1)
    return {k: v / n for k, v in totals.items()}


@torch.no_grad()
def validate(model:  DefectCVAE,
             loader: DataLoader,
             device: str,
             beta:   float = 1e-3) -> dict[str, float]:
    model.eval()
    totals = {"loss": 0.0, "mse": 0.0, "ce": 0.0, "kl": 0.0}

    for batch in loader:
        # batch = {k: v.to(device) for k, v in batch.items()}
        recon, mu, log_var = model(batch)

        B = batch.band_gap.shape[0]
        target_cloud = batch.defect_cloud.view(B, -1, batch.defect_cloud.shape[-1])
        losses = cvae_loss(recon, target_cloud, mu, log_var, beta)
        # losses = cvae_loss(recon, batch["defect_cloud"], mu, log_var, beta)
        for k in totals:
            totals[k] += losses[k].item()

    n = max(len(loader), 1)
    return {k: v / n for k, v in totals.items()}


def train(model,
          train_set,
          val_set,
          n_epochs,
          batch_size,
          lr,
          beta,
          device,
          save_path) -> dict:

    train_loader = DataLoader(train_set, batch_size=batch_size,
                              shuffle=True) # ,  collate_fn=collate_fn)
    val_loader   = DataLoader(val_set,   batch_size=batch_size,
                              shuffle=False)# , collate_fn=collate_fn)

    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=n_epochs, eta_min=lr * 0.01
    )

    history = {"train_loss": [], "val_loss": [], "kl": [], "mse": []}
    best_val_loss = float("inf")

    for epoch in range(1, n_epochs + 1):
        # KL warm-up: linearly ramp beta over first 20 epochs
        current_beta = beta * min(1.0, epoch / 20.0)

        train_stats = train_one_epoch(model, train_loader, optimizer,
                                       device, current_beta)
        val_stats   = validate(model, val_loader, device, current_beta)
        scheduler.step()

        history["train_loss"].append(train_stats["loss"])
        history["val_loss"].append(val_stats["loss"])
        history["kl"].append(train_stats["kl"])
        history["mse"].append(train_stats["mse"])

        # Checkpoint best model
        if val_stats["loss"] < best_val_loss:
            best_val_loss = val_stats["loss"]
            torch.save(model.state_dict(), save_path)
            best_marker = " ← best"
        else:
            best_marker = ""

        if epoch % 10 == 0 or epoch == 1:
            print(f"Epoch {epoch:4d}/{n_epochs} | "
                  f"train {train_stats['loss']:.4f} | "
                  f"val {val_stats['loss']:.4f} | "
                  f"KL {train_stats['kl']:.4f} | "
                  f"MSE {train_stats['mse']:.4f}")

    print(f"\nBest val loss: {best_val_loss:.4f}  (saved to {save_path})")
    return history

model = DefectCVAE()

device  = "cuda" if torch.cuda.is_available() else "cpu"

history = train(
    model=model,
    train_set=train_dataset,
    val_set=val_dataset,
    n_epochs=20,
    batch_size=8,
    lr=3e-4,
    beta=1e-3,
    device=device,
    save_path="cvae_defect_smoketest.pt",
)

Epoch    1/20 | train 0.8665 | val 0.8645 | KL 0.0062 | MSE 0.1718
Epoch   10/20 | train 0.6374 | val 0.6318 | KL 1.1805 | MSE 0.0323
Epoch   20/20 | train 0.5229 | val 0.5532 | KL 3.2525 | MSE 0.0315

Best val loss: 0.5529  (saved to cvae_defect_smoketest.pt)


## Inverse Design

In [83]:

def apply_defects_to_structure(pristine, defect_points):

    COORD_TOL = 0.3   # fractional-coordinate tolerance — generous for generated coords

    defective = pristine.copy()
    vacancy_indices = []

    for dp in defect_points:
        defect_frac = np.array(dp["fractional_coords"]) % 1.0  # wrap to [0,1]
        dtype       = dp["defect_type"]
        Zd          = dp["Z_defective"]

        best_idx, best_dist = None, float("inf")
        for idx, site in enumerate(defective):
            delta = defect_frac - site.frac_coords
            delta -= np.round(delta)          # minimum image convention
            dist = np.linalg.norm(delta)
            if dist < best_dist:
                best_dist, best_idx = dist, idx

        if best_idx is None or best_dist > COORD_TOL:
            print(f"  Warning: no site found within tol={COORD_TOL} for "
                f"{dtype} at {np.round(defect_frac, 3)} "
                f"(closest={best_dist:.3f}). Skipping this defect.")
            continue

        if dtype == "vacancy":
            if best_idx not in vacancy_indices:
                vacancy_indices.append(best_idx)

        elif dtype in ("substitution", "antisite") and Zd > 0:
            elem = Element.from_Z(Zd)
            defective[best_idx] = PeriodicSite(
                elem,
                defective[best_idx].frac_coords,
                defective[best_idx].lattice,
            )

    # Remove vacancies in reverse order so indices stay valid
    for idx in sorted(vacancy_indices, reverse=True):
        defective.remove_sites([idx])

    return defective


def inverse_design(model, pristine, target_band_gap, n_candidates, device):
    model.eval()
    model = model.to(device)

    # Generate defect clouds
    defect_clouds = model.generate(target_band_gap, n_candidates, device)

    # Apply each cloud to the pristine structure
    candidates = []
    for i, cloud in enumerate(defect_clouds):
        if not cloud:
            print(f"  Candidate {i+1}: no defects generated (empty cloud), skipping.")
            continue
        try:
            defective = apply_defects_to_structure(pristine, cloud)
            candidates.append(defective)
            print(f"  Candidate {i+1}: {len(cloud)} defect(s) | "
                f"{len(defective)} atoms | "
                f"types: {[d['defect_type'] for d in cloud]}")
        except Exception as e:
            print(f"  Candidate {i+1}: failed to apply defects ({e})")

    return candidates


candidates = inverse_design(
    model           = model,
    pristine        = reference_structure,
    target_band_gap = 1.5,
    n_candidates    = 3,
    device          = device,
)

  Candidate 1: 25 defect(s) | 128 atoms | types: ['substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution']
  Candidate 2: 25 defect(s) | 128 atoms | types: ['substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution', 'substitution']
  Candidate 3: 25 defect(s) | 128 atoms | types: ['substitution', 'substitution', 'substitution', 's

In [84]:
print(candidates[0])

Full Formula (Ga1 B60 N60 O4 F1 Ne2)
Reduced Formula: GaB60N60O4FNe2
abc   :  20.099428  20.099428  20.000000
angles:  90.000000  90.000000 120.000012
pbc   :       True       True       True
Sites (128)
  #  SP           a         b         c
---  ----  --------  --------  --------
  0  B     0.041667  0.083333  0.192682
  1  B     0.041667  0.208333  0.192682
  2  B     0.041667  0.333333  0.192682
  3  B     0.041667  0.458333  0.192682
  4  B     0.041667  0.583333  0.192682
  5  B     0.041667  0.708333  0.192682
  6  B     0.041667  0.833333  0.192682
  7  B     0.041667  0.958333  0.192682
  8  N     0.166667  0.083333  0.192682
  9  B     0.166667  0.208333  0.192682
 10  B     0.166667  0.333333  0.192682
 11  B     0.166667  0.458333  0.192682
 12  B     0.166667  0.583333  0.192682
 13  B     0.166667  0.708333  0.192682
 14  B     0.166667  0.833333  0.192682
 15  B     0.166667  0.958333  0.192682
 16  B     0.291667  0.083333  0.192682
 17  O     0.291667  0.208333  0.192

/usr/local/lib/python3.12/dist-packages/pymatgen/core/composition.py:1398: UserWarning: No Pauling electronegativity for Ne. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  return sorted(sym, key=lambda x: [float("inf") if math.isnan(e_neg := get_el_sp(x).X) else e_neg, x])
/usr/local/lib/python3.12/dist-packages/pymatgen/core/composition.py:1376: UserWarning: No Pauling electronegativity for Ne. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  if len(syms) >= 3 and get_el_sp(syms[-1]).X - get_el_sp(syms[-2]).X < 1.65:
